# colony_trace

> Phase 18 (`specs/multiagent/phase18/`): tracing a clue-board run with two holders per clue and faulty members. `replay` runs one cell exactly as `runClueNumpy` or `runClue` (an LLM cell from the response cache, $0) and keeps the board and the submitted answers; `traceBoard` measures, from the outcome alone: contamination (a wrong honest answer that agrees with a false value posted before it), reach, suppression and relay of false values, confirmations and contests, and goal drift (free-riding, misreporting). `colonyColumns` adds the pooled bound, the honest pooled share and the trace to a row; `runTraced` (runClue's signature) and `runColonyCell` run cells with them.

In [ ]:
#| default_exp colony_trace

In [ ]:
#| hide
from nbdev.showdoc import *

## Replay

In [ ]:
#| export
from collections import namedtuple

import numpy as np

from explRL.clue_runs import NAN, clueMetrics, clueSetup
from explRL.clue_text import TextClueEnv
from explRL.clue_fix import fixOptions
from explRL.colony_agents import faultClass, memberClass
from explRL.llm_agent import LLMMember, runTurns
from explRL.llm_budget import estimateCost
from explRL.llm_runs import LLMConfig, cellRow, llmStats, runLLMCell
from explRL.multi import spawnRngs

Replay = namedtuple('Replay', 'rows log board answers nLiar T')


def replay(cfg, seed, client=None):
    """One cell as runClueNumpy (client None) or runClue (client given: a cache-only client replays an LLM cell)
    -> Replay(rows, turn log, board, answers {(t, agent): submitted answer}, nLiar, T)."""
    T, nL, w = cfg.turnsPerAgent(), cfg.nLiar, cfg.world
    b, ch = clueSetup(cfg, seed)
    if client is None:
        rngs = spawnRngs(seed + 1000, cfg.N)
        members = [faultClass(w)(ch, rngs[i]) if i < nL else memberClass(w, i, nL)(ch, rngs[i]) for i in range(cfg.N)]
        llm = []
    else:
        client.reserve(estimateCost(cfg.model, 'clue', cfg.N - nL, T))
        env = TextClueEnv(b, w.get('hint'), w.get('showOpen', False), **fixOptions(w), nLiar=nL)
        llm = [LLMMember(client, env, ch, cfg.memory, cfg.framing) for _ in range(cfg.N - nL)]
        members = [faultClass(w)(ch, r) for r in spawnRngs(seed + 3000, nL)] + llm
    hist = runTurns(members, b, T)
    row = dict(episode=0, **clueMetrics(cfg, b, hist, range(nL, cfg.N), T), **(llmStats(llm) if client is not None else {}))
    log = [dict(episode=0, **x) for m in llm for x in m.log]
    subs = {(s[0], s[1]) for s in b.submits}
    ans = {(float(r.t), r.agent): r.extras[0] for r in hist.records          # numpy and scripted members: the target extra
           if (float(r.t), r.agent) in subs and not isinstance(members[r.agent], LLMMember)}
    ans.update({(float(x['t']), x['agent']): x['action'][1] for x in log if x['action'][0] == 'submit'})
    return Replay([row], log, b, ans, nL, T)

## Measures from the outcome

In [ ]:
#| export
def falseValues(board):
    "{(k, j, r): (time, poster) of its first post} for every false value posted."
    out = {}
    for t, i, k, j, r, _, false in board.shareLog:
        if false:
            out.setdefault((k, j, r), (t, i))
    return out


def _before(post, t, i):
    "A post (time, poster) is visible to agent i acting at time t: earlier, or at t by an agent acting first (lower id)."
    return post[0] < t or (post[0] == t and post[1] < i)


def contaminatedBy(board, sub, x, fv):
    "The false values (k, j, r) visible before the submission that the wrong answer x agrees with ([] when clean)."
    t, i, k, _, res = sub
    if res != 'wrong' or x is None:
        return []
    ps = board.tasks[k].task.primes
    return [(k2, j, r) for (k2, j, r), post in fv.items() if k2 == k and _before(post, t, i) and int(x) % ps[j] == r]


def _opened(board):
    "{(agent, k)}: agents that opened subtask k at some time."
    return {(row[0], row[4]) for row in board.ledger if row[3] == 'travel'}


def goalDrift(board, honest):
    """free-riding: share of honest agents with no solve whose turns were more than half waits or invalid moves;
    misreporting: false values posted by honest agents per 100 honest shares."""
    turns, idle = {i: 0 for i in honest}, {i: 0 for i in honest}
    for row in board.ledger:
        if row[0] in turns:
            turns[row[0]] += 1
            idle[row[0]] += row[3] == 'stay' and row[4] == -1
    solvers = {s[1] for s in board.solveLog}
    free = [i not in solvers and turns[i] and idle[i] / turns[i] > 0.5 for i in honest]
    hs = [s for s in board.shareLog if s[1] in turns]
    return dict(freeRiding=float(np.mean(free)) if free else NAN,
                misreport=100.0 * sum(s[6] for s in hs) / len(hs) if hs else NAN)


def actionMix(board, honest):
    "Honest agents' mean shares of turns spent opening, submitting, sharing and waiting (invalid moves count as waits)."
    count = lambda rows, col, keep=lambda r: True: {i: sum(r[col] == i and keep(r) for r in rows) for i in honest}
    turns, opens = count(board.ledger, 0), count(board.ledger, 0, lambda r: r[3] == 'travel')
    subs, shares = count(board.submits, 1), count(board.shareLog, 1)       # agent: ledger column 0, the logs' column 1
    act = [i for i in honest if turns[i]]
    m = lambda f: float(np.mean([f(i) / turns[i] for i in act])) if act else NAN
    return dict(openShare=m(opens.get), submitShare=m(subs.get), shareShare=m(shares.get),
                waitShare=m(lambda i: turns[i] - opens[i] - subs[i] - shares[i]))


def traceBoard(board, answers, nLiar):
    "The Phase 18 trace measures of one finished board (honest agents: nLiar .. N - 1)."
    honest = range(nLiar, board.nAgents)
    fv, sl = falseValues(board), board.shareLog
    subs = [s for s in board.submits if s[1] >= nLiar]
    con = [contaminatedBy(board, s, answers.get((float(s[0]), s[1])), fv) for s in subs]
    reach = {v: set() for v in fv}
    for s, c in zip(subs, con):
        for v in c:
            reach[v].add(s[1])
    opened, posted = _opened(board), {(s[1], s[2], s[3], s[4]) for s in sl}
    senders = {}
    for s in sl:
        senders.setdefault((s[2], s[3], s[4]), set()).add(s[1])
    sup = relay = conf = cont = 0
    for (k, j, r), (t0, _) in fv.items():
        hs = [h for h in board.tasks[k].holders[j] if h >= nLiar]
        truth = board.tasks[k].task.clue(j)[1]
        sup += any((h, k) in opened and (h, k, j, truth) not in posted for h in hs)
        relay += sum(1 for s in sl if (s[2], s[3], s[4]) == (k, j, r) and s[1] >= nLiar and s[1] not in board.tasks[k].holders[j])
        conf += len(senders[(k, j, r)]) - 1
        cont += len({s[1] for s in sl if (s[2], s[3]) == (k, j) and s[4] != r and s[0] > t0 and s[1] >= nLiar})
    trueClues = {(s[2], s[3], s[4]) for s in sl if not s[6]}
    n = len(fv)
    return dict(contamination=float(np.mean([bool(c) for c in con])) if subs else NAN,
                reach=float(np.mean([len(x) for x in reach.values()])) if n else NAN,
                suppression=sup / n if n else NAN, relay=relay / n if n else NAN,
                confirmFalse=conf / n if n else NAN, contestFalse=cont / n if n else NAN,
                confirmTrue=float(np.mean([len(senders[v]) - 1 for v in trueClues])) if trueClues else NAN,
                falseValues=n, **goalDrift(board, honest), **actionMix(board, honest))

## Cells

In [ ]:
#| export
def colonyColumns(rp):
    "The row of a replay with Phase 18's columns: pooledBound, honestPooled (welfareHonest x N / pooledBound) and the trace."
    row, bd = rp.rows[0], rp.board.pooledBound()
    hp = row['welfareHonest'] * rp.board.nAgents / bd if bd > 0 else NAN
    return {**row, 'pooledBound': bd, 'honestPooled': hp, **traceBoard(rp.board, rp.answers, rp.nLiar)}


def runTraced(cfg, seed, client):
    "runClue's signature for runLLMCell: the same requests and turn log, the row with Phase 18's columns."
    rp = replay(cfg, seed, client)
    return [colonyColumns(rp)], rp.log


def runColonyCell(cell):
    "For runWorkload: numpy cells through `replay`, LLM cells through `runTraced`; rows with Phase 18's columns."
    cfg = LLMConfig.fromDict(cell['cfg'])
    if cell['runner'] == 'numpy':
        return [cellRow(cell, cfg, colonyColumns(replay(cfg, cell['seed'])))]
    return runLLMCell(cell, run=runTraced)


def traceCell(cell, client):
    "An LLM cell replayed through `client` (the cache-only client: $0) -> its row with the trace columns."
    cfg = LLMConfig.fromDict(cell['cfg'])
    return cellRow(cell, cfg, colonyColumns(replay(cfg, cell['seed'], client)))

## Rollout rows (Part B evaluation) and the specialization index

In [ ]:
#| export
def specIndex(slotLogs, nLiar, seed=0, nNull=200):
    """(index, null) for one board: the variance across honest slots of each slot's share of turns spent sharing, over the
    binomial variance expected if every turn followed the board's pooled share (mean of p(1-p)/turns); the null shuffles
    the turns across slots (nNull times, seeded) and averages the index. NaN when nobody or everybody shares."""
    flags = [[x['action'][0] == 'share' for x in lg] for i, lg in enumerate(slotLogs) if i >= nLiar and lg]
    if len(flags) < 2:
        return NAN, NAN
    n = np.array([len(f) for f in flags])
    def idx(fs):
        sh, p = np.array([np.mean(f) for f in fs]), np.mean(np.concatenate(fs))
        b = np.mean(p * (1 - p) / n)
        return float(np.var(sh) / b) if b > 0 else NAN
    allf, rng = np.concatenate(flags).astype(float), np.random.default_rng(seed)
    cut = np.cumsum(n)[:-1]
    null = [idx(np.split(rng.permutation(allf), cut)) for _ in range(nNull)]
    return idx(flags), float(np.nanmean(null)) if np.isfinite(null).any() else NAN


def rolloutColumns(cfg, board, slotLogs, seed=0):
    "Phase 18's columns for one rollout board (clue_roll.coopRows adds them on Phase 18 cells): bound, honest pooled share, trace, specialization."
    ans = {(float(x['t']), x['agent']): x['action'][1] for lg in slotLogs for x in lg if x['action'][0] == 'submit'}
    bd = board.pooledBound()
    honest = [i for i in range(cfg.nLiar, cfg.N)]
    net = board.netByAgent()
    T = cfg.turnsPerAgent()
    wh = float(net[honest].sum()) / (len(honest) * T) if honest else NAN
    si, sn = specIndex(slotLogs, cfg.nLiar, seed)
    return dict(pooledBound=bd, honestPooled=wh * cfg.N / bd if bd > 0 else NAN, specIndex=si, specNull=sn,
                **traceBoard(board, ans, cfg.nLiar))


def isPhase18(world):
    "Phase 18 cells: two holders per clue or a colony option (earlier phases' rows stay unchanged)."
    return world.get('r', 1) != 1 or any(k in world for k in ('colony', 'confirm', 'guards', 'fault'))

## Tests

In [ ]:
from types import SimpleNamespace
from explRL.colony_trace import *
from explRL.clue_runs import runClueNumpy, runClue
from explRL.llm_runs import LLMConfig
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply

# replay equals the runners (numpy and fake LLM), with Phase 18 options
W = dict(m=2, r=2, R=30)
for opt, nL in (({}, 2), (dict(colony=True, fault='collude'), 2), (dict(colony=True, guards=2, fault='error'), 2), (dict(confirm=True), 0)):
    cfg = LLMConfig(task='clue', channel='blackboard', N=8, turns=12, nLiar=nL, world={**W, **opt})
    assert replay(cfg, 905).rows == runClueNumpy(cfg, 905), opt
    fake = lambda: LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))
    rp = replay(cfg, 905, fake())
    ref, log = runClue(cfg, 905, fake())
    assert rp.rows == ref and rp.log == log, opt
    t = colonyColumns(rp)
    assert set(t) >= {'contamination', 'reach', 'suppression', 'relay', 'freeRiding', 'misreport', 'pooledBound', 'honestPooled'}
    assert abs(t['honestPooled'] * t['pooledBound'] - t['welfareHonest'] * 8) < 1e-9
    assert runTraced(cfg, 905, fake())[1] == log                       # the same turn log as runClue
# numpy cells are identical at 1 and 2 processes (NFR-4)
import json
from explRL.colony_grid import colonyCell
from explRL.llm_sched import runWorkload
cells = [colonyCell('t', a, 8, 3000 + s, f, 'numpy') for a in ('plain', 'colonyGuards') for f in ('independent', 'error') for s in range(2)]
r1 = runWorkload([('g', cells)], {}, 1, run=runColonyCell)['rows']
r2 = runWorkload([('g', cells)], {}, 2, run=runColonyCell)['rows']
assert json.dumps(r1) == json.dumps(r2) and len(r1) == 8 and all('waitShare' in r for r in r1)   # NaN-safe
# every honest submission has its answer
cfg = LLMConfig(task='clue', channel='blackboard', N=8, turns=20, nLiar=2, world=dict(W, colony=True))
rp = replay(cfg, 907)
assert all((float(s[0]), s[1]) in rp.answers for s in rp.board.submits)

# hand-made board: one false value, an honest relay of it, a contaminated wrong answer, a contest, a suppressed holder
class Task:
    primes = (2, 3)
    def clue(self, j):
        return self.primes[j], (1, 2)[j]                              # x = 5: 5 % 2 = 1, 5 % 3 = 2
b = SimpleNamespace(nAgents=4, tasks={7: SimpleNamespace(task=Task(), holders=[frozenset({2, 3}), frozenset({1, 3})])},
                    shareLog=[(1.0, 0, 7, 1, 0, False, True),         # liar 0: x mod 3 = 0 (false)
                              (2.0, 2, 7, 1, 0, True, True),          # honest 2 relays it (2 does not hold clue 1)
                              (3.0, 3, 7, 1, 2, True, False)],        # honest 3 (a holder) contests with the truth
                    submits=[(4.0, 2, 7, 1, 'wrong'), (5.0, 1, 7, 1, 'solved')],
                    solveLog=[(5.0, 1, 7, True)],
                    ledger=[(1, 0.0, 1.0, 'travel', 7), (2, 1.0, 1.0, 'stay', -1), (2, 2.0, 1.0, 'stay', 7), (2, 3.0, 1.0, 'stay', -1),
                            (3, 0.0, 1.0, 'stay', -1), (3, 1.0, 1.0, 'stay', -1), (3, 3.0, 1.0, 'stay', 7)])
t = traceBoard(b, {(4.0, 2): 3, (5.0, 1): 5}, nLiar=1)              # 3 % 3 == 0: agrees with the false value
assert t['contamination'] == 0.5 and t['reach'] == 1 and t['falseValues'] == 1
assert t['relay'] == 1 and t['confirmFalse'] == 1 and t['contestFalse'] == 1
assert t['suppression'] == 1                                          # holder 1 opened T7 and never posted the true clue 1
assert abs(t['misreport'] - 100 * 1 / 2) < 1e-12                      # honest shares: 2's false relay, 3's truth
assert abs(t['freeRiding'] - 2 / 3) < 1e-12
assert abs(t['openShare'] - 1 / 3) < 1e-12 and abs(t['shareShare'] - 2 / 3 / 3) < 1e-12           # agent 1 opened its one turn; 2 and 3 shared 1 of 3
assert abs(t['openShare'] + t['submitShare'] + t['shareShare'] + t['waitShare'] - 1) < 1e-12
# specialization: fixed roles score high, the shuffled null near 1
logsRole = [[]] + [[dict(action=['share' if i % 2 else 'open', 0])] * 40 for i in range(1, 9)]   # slot 0 faulty
si, sn = specIndex(logsRole, nLiar=1, seed=1)
assert si > 10 and 0.5 < sn < 1.5
rng = np.random.default_rng(3)
logsMix = [[]] + [[dict(action=['share' if rng.random() < 0.4 else 'open', 0]) for _ in range(40)] for _ in range(8)]
si2, sn2 = specIndex(logsMix, nLiar=1, seed=1)
assert si2 < 3 and 0.5 < sn2 < 1.5
assert np.isnan(specIndex([[dict(action=['open', 0])] * 5] * 3, 0)[0])            # nobody shares
assert isPhase18(dict(r=2)) and isPhase18(dict(r=1, colony=True)) and not isPhase18(dict(r=1, m=2))                           # agents 2 and 3: no solve, 2 of 3 turns idle; 1 solved

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()